In [11]:
import threading
import time
from collections import deque

class BlockingQueue:
    def __init__(self, capacity):
        self.capacity = capacity
        self._items = deque()
        self._lock = threading.Lock()
        self._not_full = threading.Condition(self._lock)
        self._not_empty = threading.Condition(self._lock)
        self._close = False

    def close(self):
        with self._lock:
            self._close = True
            self._not_empty.notify_all()
            self._not_full.notify_all()

    def put(self, item, timeout = None):
        with self._not_full:
            deadline = None if timeout is None else time.monotonic() + timeout
            while len(self._items) == self.capacity:
                if self._close:
                    raise RuntimeError('Queue closed')
                remaining = None if deadline is None else deadline - time.monotonic()
                if remaining is not None and remaining <= 0:
                    raise TimeoutError("Time Elapsed: Queue full")
                self._not_full.wait(remaining)
            self._items.append(item)
            self._not_empty.notify()

    def get(self, timeout = None):
        with self._not_empty:
            deadline = None if timeout is None else time.monotonic() + timeout
            while len(self._items) == 0:
                if self._close:
                    return None
                remaining = None if deadline is None else deadline - time.monotonic()
                if remaining is not None and remaining <= 0:
                    raise TimeoutError("Time Elapsed: Queue empty")
                self._not_empty.wait(remaining)
            item = self._items.popleft()
            self._not_full.notify()
            return item

In [12]:
import time, random

q = BlockingQueue(capacity=3)

def producer(pid, count):
    for i in range(count):
        item = f"P{pid}-{i}"
        q.put(item)
        print(f"[producer {pid}] put {item} (size={len(q._items)})")
        time.sleep(random.uniform(0.05, 0.2))

def consumer(cid):
    while True:
        item = q.get()
        if item is None: break
        print(f"        [consumer {cid}] got {item}")
        time.sleep(random.uniform(0.1, 0.3))   # slower → forces producers to block

producers = [threading.Thread(target=producer, args=(1, 5)),
             threading.Thread(target=producer, args=(2, 5))]
consumers = [threading.Thread(target=consumer, args=(1,)),
             threading.Thread(target=consumer, args=(2,))]

for t in producers + consumers: t.start()
for t in producers: t.join()   # wait until ALL producers finish
q.close()                       # now it's safe to close
for t in consumers: t.join()

[producer 1] put P1-0 (size=1)
[producer 2] put P2-0 (size=2)
        [consumer 1] got P1-0
        [consumer 2] got P2-0
[producer 2] put P2-1 (size=1)
[producer 1] put P1-1 (size=2)
        [consumer 2] got P2-1
        [consumer 1] got P1-1
[producer 1] put P1-2 (size=1)
[producer 2] put P2-2 (size=2)
        [consumer 1] got P1-2
        [consumer 2] got P2-2
[producer 2] put P2-3 (size=1)
[producer 1] put P1-3 (size=2)
        [consumer 2] got P2-3
        [consumer 1] got P1-3
[producer 2] put P2-4 (size=1)
[producer 1] put P1-4 (size=2)
        [consumer 1] got P2-4
        [consumer 2] got P1-4


In [ ]:
from queue import PriorityQueue

pq = PriorityQueue()